## Silver - dim_cliente

### Transformações da camada bronze para silver

####Importando bibliotecas

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.functions import trim, col
from pyspark.sql.types import StringType

### dim_cliente
####Consultando dados da tabela bronze

In [0]:
df_cliente = spark.table("dev_credit_risk.bronze.dim_cliente")
display(df_cliente)

#### Início das transformações
#### Retirando os espaços em branco das colunas de tipo string.

In [0]:
for field in df_cliente.schema.fields:
  if isinstance(field.dataType, StringType):
    df_cliente = df_cliente.withColumn(field.name, trim(col(field.name)))

df_cliente.display()

#### Normalizando os valores

In [0]:
df_cliente = (
    df_cliente
    .withColumn(
        "sexo",
        F.when(F.upper(F.col("sexo")).startswith("M"), "Masculino")
         .when(F.upper(F.col("sexo")).startswith("F"), "Feminino")
        .otherwise(F.col("sexo"))
    )
    .withColumn(
        "estadoCivil",
        F.when(F.upper(F.col("estadoCivil")).startswith("S"), "Solteiro")
         .when(F.upper(F.col("estadoCivil")).startswith("C"), "Casado")
        .otherwise(F.col("estadoCivil"))
    )
    .withColumn(
        "UF",
        F.when(F.upper(F.col("UF")).startswith("P"), "PE")
         .when(F.upper(F.col("UF")).startswith("S"), "SP")
        .otherwise(F.col("UF"))
    )
)
df_cliente.display()

####Checando o domínio. Valores únicos após normalização

In [0]:
#checando os valores únicos.
display(df_cliente.select("sexo","estadoCivil","uf").distinct())

####Deletando coluna(s) desnecessárias

In [0]:
df_cliente = df_cliente.drop("cidade")

###Padronizando os nomes das colunas

In [0]:
RENAME_MAP = {
    "id_cliente": "id_cliente",
    "NomeCliente": "nome",
    "idade": "idade",
    "sexo": "sexo",
    "estadoCivil": "estado_civil",
    "FaixaRenda": "faixa_renda",
    "UF": "uf"
}

####Renomeando as colunas

In [0]:
for old_name, new_name in RENAME_MAP.items():
    df_cliente = df_cliente.withColumnRenamed(old_name, new_name)


####Exibindo o dataframe pronto para ser inserido na tabela delta silver

In [0]:
df_cliente.display()

####Inserindo os dados na tabela bronze silver

In [0]:
(df_cliente.write
    .mode("overwrite")
    .format("delta")
    .saveAsTable("dev_credit_risk.silver.dim_cliente")
)

####Consultando os dados com SQL no schema tabela delta silver

In [0]:
%sql
select * from dev_credit_risk.silver.dim_cliente